In [ ]:
import pandas as pd

df = pd.read_csv("combined_results.csv")


,model_path,folder,val_loss,val_acc,time_per_image,throughput,nums_trainable_parameters
0,model_weights/final_checkpoint_resnet18_batch_...,masked,1.370018,0.635802,0.000070,14233.909027,51813
1,model_weights/final_checkpoint_resnet18_batch_...,clean,1.191958,0.678733,0.000051,19683.366444,51813
2,model_weights/final_checkpoint_resnet18_batch_...,downsampled,3.599344,0.229465,0.000049,20258.121235,51813
3,model_weights/final_checkpoint_resnet18_batch_...,blur_medium,2.459206,0.397307,0.000052,19417.276592,51813
4,model_weights/final_checkpoint_resnet18_batch_...,blur_little,1.525196,0.595842,0.000051,19674.948674,51813


In [38]:
# num of trained parameters vs improvement over baseline

df_grouped = df.groupby("model_path").agg({
    "val_loss": "mean",
    "val_acc": "mean",
    "throughput": "mean",
    "nums_trainable_parameters": "mean"
}).reset_index()

df_over_baseline = df_grouped.copy()
baseline_acc_res = df_grouped[df_grouped["model_path"].str.contains("resnet18_linear_probe")]["val_acc"]
baseline_acc_eff = df_grouped[df_grouped["model_path"].str.contains("efficientnet_linear_probe")]["val_acc"]



resnet_df = df_over_baseline[df_over_baseline["model_path"].str.contains("resnet")]
resnet_df["acc_over_baseline"] = resnet_df["val_acc"] - baseline_acc_res.values[0]
efficientnet_df = df_over_baseline[df_over_baseline["model_path"].str.contains("efficientnet")]
efficientnet_df["acc_over_baseline"] = efficientnet_df["val_acc"] - baseline_acc_eff.values[0]

In [39]:
resnet_df

,model_path,val_loss,val_acc,throughput,nums_trainable_parameters,acc_over_baseline
4,model_weights/final_checkpoint_resnet18_batch_...,1.948420,0.520950,18913.692941,51813.0,0.094026
5,model_weights/final_checkpoint_resnet18_custom...,1.688231,0.571465,9786.243891,194469.0,0.144541
6,model_weights/final_checkpoint_resnet18_linear...,2.413330,0.426924,19428.036002,51813.0,0.000000
7,model_weights/final_checkpoint_resnet18_lora.pt,2.040917,0.513630,18576.245452,572517.0,0.086706
8,model_weights/final_checkpoint_resnet18_task_s...,1.960891,0.521809,16555.919997,139933.0,0.094884


In [40]:

import altair as alt

res_chart = alt.Chart(resnet_df).mark_point().encode(
    x="nums_trainable_parameters",
    y="acc_over_baseline",
    color="model_path"
)

efficientnet_chart = alt.Chart(efficientnet_df).mark_point().encode(
    x="nums_trainable_parameters",
    y="acc_over_baseline",
    color="model_path"
)

res_chart | efficientnet_chart

alt.HConcatChart(...)

analysis to be done
- Compare acc across all models
- Compare between EfficientNet and ResNet models
- Between different test sets
- Compare numbers of parameters vs accuracy over baseline (linear_probe) for all models
- Compare inference time per image for all models
- Visualizations showing trade-offs between performance and model efficiency, such as bubble plots of performance vs FLOPs2
or wall-clock time, with bubble  size proportional to trainable parameter count. 
- Similar plots should be created for inference speed vs performance.

Make nice charts out of these.
Make grad cam files


